<a href="https://colab.research.google.com/github/jasonmcgovern/gallery/blob/main/hubSpotImpersonator_brandSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Brand Sentiment — v2

Measures **how positively ChatGPT talks about a brand when people ask about it.**

**How a run works**
1. Builds a prompt set: a fixed **core** set of brand-agnostic questions (same for every brand and every run, so scores are comparable) plus **generated** brand-specific questions (varies run to run, like real usage).
2. Asks ChatGPT each question (optionally with **web search**, like ChatGPT in the wild).
3. A separate, cheap **judge** model scores each answer from −2 to +2, tags aspects, and pulls verbatim evidence.
4. Reports a **Sentiment Index (0–100)**, the v1-style **Brand-safe rate** (% not negative), results by question intent and aspect, and an optional **themes summary**.
5. Exports CSVs you can re-upload later to **track over time** or **compare brands**.

**Setup:** upload a text file named `openai_api_key.txt` containing only your OpenAI key, using the 📁 **Files** panel (left sidebar). Colab deletes uploaded files when the runtime resets, so re-upload it each session. (Colab 🔑 Secrets as `OPENAI_API_KEY` also works as a fallback.) Never paste keys into cells or commit the key file to GitHub.

Run cells top to bottom. Cells 1–2 are setup; cell 3 is where you set the brand.

In [ ]:
# @title 1 · Install & imports
!pip -q install -U openai

import os, re, json, time, math, random, threading, uuid
from datetime import datetime, timezone
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse

import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI

API_KEY_FILE = "openai_api_key.txt"  # upload via the 📁 Files panel (left sidebar)

def _load_api_key():
    """Key lookup order: uploaded openai_api_key.txt → Colab Secrets → env var → prompt."""
    candidates = [API_KEY_FILE, f"/content/{API_KEY_FILE}",
                  f"/content/drive/MyDrive/{API_KEY_FILE}"]
    for path in dict.fromkeys(candidates):
        if os.path.isfile(path):
            with open(path) as f:
                key = f.read().strip().strip('"').strip("'")
            if key:
                print(f"✓ Using API key from {path}")
                return key
            print(f"⚠️ {path} is empty — checking other sources")
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            print("✓ Using API key from Colab Secrets")
            return key
    except Exception:
        pass
    if os.environ.get("OPENAI_API_KEY"):
        print("✓ Using API key from environment")
        return os.environ["OPENAI_API_KEY"]
    print(f"No {API_KEY_FILE} found. Upload it in the 📁 Files panel and re-run this cell, "
          "or paste the key below.")
    from getpass import getpass
    return getpass("OpenAI API key (not stored): ").strip()

client = OpenAI(api_key=_load_api_key(), max_retries=4, timeout=180)
print("✓ OpenAI client ready")

In [ ]:
# @title 2 · Pricing table (edit if OpenAI changes prices)
# USD per 1M tokens. "cached" = cached input rate.
# Verified Oct 2026 for gpt-6 family. chat-latest is estimated from measured
# costs (~$0.058 per web-search answer, Oct 2026) — the run tracks real usage anyway.
PRICING = {
    "chat-latest":  {"input": 5.00, "cached": 0.50, "output": 30.00},
    "gpt-6-sol":    {"input": 2.00, "cached": 0.20, "output": 10.00},
    "gpt-6.1-sol":  {"input": 2.00, "cached": 0.20, "output": 10.00},
    "gpt-6-luna":   {"input": 0.10, "cached": 0.01, "output": 0.50},
    "gpt-5.6-luna": {"input": 0.20, "cached": 0.02, "output": 1.20},
}
WEB_SEARCH_PER_CALL = 0.01   # $10 per 1,000 search calls

def price_for(model):
    if model in PRICING:
        return PRICING[model]
    for k, v in PRICING.items():          # e.g. dated snapshots like chat-latest-2026-10-07
        if model.startswith(k):
            return v
    print(f"⚠️ No price for '{model}' — using chat-latest rates as a conservative guess")
    return PRICING["chat-latest"]

def usage_cost(model, usage, search_calls=0):
    p = price_for(model)
    inp = getattr(usage, "input_tokens", 0) or 0
    out = getattr(usage, "output_tokens", 0) or 0
    det = getattr(usage, "input_tokens_details", None)
    cached = (getattr(det, "cached_tokens", 0) or 0) if det else 0
    return ((inp - cached) * p["input"] + cached * p["cached"] + out * p["output"]) / 1e6 \
           + search_calls * WEB_SEARCH_PER_CALL

In [ ]:
# @title 3 · Configure the run
BRAND = "Your Brand Name"  # @param {type:"string"}
# Optional context — helps generated questions & the judge avoid confusing similarly named companies
CATEGORY = "orthodontist"  # @param {type:"string"}
# Optional — for local businesses, makes web search results local (leave blank for national/global brands)
MARKET_CITY = ""  # @param {type:"string"}
MARKET_REGION = ""  # @param {type:"string"}
MARKET_COUNTRY = "US"  # @param {type:"string"}

ANALYSIS_DEPTH = 50  # @param [20, 50, 100] {type:"raw"}
SAMPLES_PER_PROMPT = 1  # @param {type:"slider", min:1, max:5, step:1}
USE_WEB_SEARCH = True  # @param {type:"boolean"}

ANSWER_MODEL = "chat-latest"  # @param ["chat-latest", "gpt-6-sol", "gpt-6-luna"] {allow-input: true}
JUDGE_MODEL = "gpt-6-luna"  # @param ["gpt-6-luna", "gpt-6-sol"] {allow-input: true}
WRITER_MODEL = "gpt-6-sol"  # @param ["gpt-6-sol", "gpt-6-luna"] {allow-input: true}

MAX_BUDGET_USD = 3.00  # @param {type:"number"}
INCLUDE_THEME_SUMMARY = True  # @param {type:"boolean"}
# Reuse a previously exported *_promptset.csv instead of generating new questions
REUSE_PROMPT_SET = False  # @param {type:"boolean"}
MAX_WORKERS = 6  # @param {type:"slider", min:1, max:12, step:1}

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M") + "-" + uuid.uuid4().hex[:4]
RUN_DATE = datetime.now(timezone.utc).isoformat(timespec="seconds")
BRAND_SLUG = re.sub(r"[^a-z0-9]+", "-", BRAND.lower()).strip("-")
print(f"Run {RUN_ID} · {BRAND} · depth {ANALYSIS_DEPTH} × {SAMPLES_PER_PROMPT} sample(s) · "
      f"{ANSWER_MODEL}{' + web search' if USE_WEB_SEARCH else ''}")

In [ ]:
# @title 4 · Cost estimate & budget check
# Typical token usage per call (from measured runs). Search answers read ~10.7k tokens of results.
EST = {
    "answer_search":   {"input": 10700, "cached_share": 0.40, "output": 700, "search_calls": 1},
    "answer_nosearch": {"input": 60,    "cached_share": 0.0,  "output": 700, "search_calls": 0},
    "judge":           {"input": 1900,  "cached_share": 0.30, "output": 450, "search_calls": 0},
    "generate":        {"input": 900,   "cached_share": 0.0,  "output": 4500, "search_calls": 0},
    "summary":         {"input": 25000, "cached_share": 0.0,  "output": 2500, "search_calls": 0},
}

def est_call_cost(model, kind):
    e, p = EST[kind], price_for(model)
    cached = e["input"] * e["cached_share"]
    return ((e["input"] - cached) * p["input"] + cached * p["cached"] + e["output"] * p["output"]) / 1e6 \
           + e["search_calls"] * WEB_SEARCH_PER_CALL

def estimate_run(depth, samples, answer_model=ANSWER_MODEL, search=USE_WEB_SEARCH):
    n = depth * samples
    per_answer = est_call_cost(answer_model, "answer_search" if search else "answer_nosearch")
    parts = {
        "answers": n * per_answer,
        "judging": n * est_call_cost(JUDGE_MODEL, "judge"),
        "prompt generation": 0 if REUSE_PROMPT_SET else est_call_cost(WRITER_MODEL, "generate"),
        "theme summary": est_call_cost(WRITER_MODEL, "summary") if INCLUDE_THEME_SUMMARY else 0,
    }
    return sum(parts.values()), parts, per_answer

total, parts, per_answer = estimate_run(ANALYSIS_DEPTH, SAMPLES_PER_PROMPT)
print(f"Estimated cost: ${total:.2f}  (budget ${MAX_BUDGET_USD:.2f})")
for k, v in parts.items():
    print(f"   {k:18} ${v:.3f}")
print(f"   ≈ ${per_answer:.4f} per answer · {ANALYSIS_DEPTH * SAMPLES_PER_PROMPT} answers")

if total > MAX_BUDGET_USD:
    print("\n⚠️  Over budget. Settings that fit (most samples per option):")
    for m in dict.fromkeys([ANSWER_MODEL, "gpt-6-sol", "gpt-6-luna"]):
        for d in (20, 50, 100):
            fits = [s for s in range(1, 6) if estimate_run(d, s, m)[0] <= MAX_BUDGET_USD]
            if fits:
                print(f"   {m:12} depth {d:>3} × up to {max(fits)} sample(s) ≈ ${estimate_run(d, max(fits), m)[0]:.2f}")
    print("\nChange settings in cell 3 (or raise MAX_BUDGET_USD) and re-run cells 3–4.")
    print("The run also hard-stops if actual spend reaches the budget.")
else:
    print("\n✓ Within budget")

In [ ]:
# @title 5 · Build the prompt set
# Core questions: identical for every brand and every run → apples-to-apples tracking & brand comparisons.
CORE_TEMPLATES = [
    ("core-01", "neutral",     "What is {brand}?"),
    ("core-02", "neutral",     "Tell me about {brand}."),
    ("core-03", "neutral",     "What is {brand} known for?"),
    ("core-04", "neutral",     "Give me an overview of {brand}."),
    ("core-05", "neutral",     "Who is {brand} best for?"),
    ("core-06", "neutral",     "What should I know about {brand} before choosing them?"),
    ("core-07", "evaluative",  "Is {brand} good?"),
    ("core-08", "evaluative",  "Is {brand} worth it?"),
    ("core-09", "evaluative",  "What do customers say about {brand}?"),
    ("core-10", "evaluative",  "{brand} reviews"),
    ("core-11", "evaluative",  "Would you recommend {brand}?"),
    ("core-12", "evaluative",  "Is {brand} trustworthy?"),
    ("core-13", "evaluative",  "How is {brand}'s customer service?"),
    ("core-14", "skeptical",   "Is {brand} legit?"),
    ("core-15", "skeptical",   "What are the downsides of {brand}?"),
    ("core-16", "skeptical",   "What are common complaints about {brand}?"),
    ("core-17", "comparative", "How does {brand} compare to its competitors?"),
    ("core-18", "comparative", "What makes {brand} different from similar companies?"),
    ("core-19", "comparative", "Is {brand} better than other options?"),
    ("core-20", "comparative", "What are the best alternatives to {brand}?"),
]
INTENT_MIX = {"neutral": 0.30, "evaluative": 0.35, "skeptical": 0.15, "comparative": 0.20}
INTENTS = list(INTENT_MIX)

def _structured(model, system, user, schema_name, schema, max_output_tokens=6000):
    resp = client.responses.create(
        model=model,
        input=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        text={"format": {"type": "json_schema", "name": schema_name, "schema": schema, "strict": True}},
        max_output_tokens=max_output_tokens,
    )
    track_spend(usage_cost(resp.model or model, resp.usage))
    return json.loads(resp.output_text)

def generate_extra_prompts(n):
    quotas = {k: round(v * n) for k, v in INTENT_MIX.items()}
    quotas["evaluative"] += n - sum(quotas.values())
    schema = {"type": "object", "additionalProperties": False, "required": ["questions"],
              "properties": {"questions": {"type": "array", "items": {
                  "type": "object", "additionalProperties": False, "required": ["intent", "prompt"],
                  "properties": {"intent": {"type": "string", "enum": INTENTS},
                                 "prompt": {"type": "string"}}}}}}
    context = f"The brand is in this category: {CATEGORY}." if CATEGORY else ""
    if MARKET_CITY or MARKET_REGION:
        context += f" It operates in {', '.join(x for x in [MARKET_CITY, MARKET_REGION] if x)}."
    user = f"""Write {n} questions that real people type into ChatGPT about the brand "{BRAND}". {context}

Quotas by intent: {json.dumps(quotas)}
- neutral: wants information (what it is, services/products, how it works, locations, hours)
- evaluative: wants a judgment (is it good, worth the price, reviews, recommendations)
- skeptical: worried or doubtful (legit?, problems, lawsuits, hidden costs) — phrased as a real worried customer would, not as a researcher
- comparative: weighing it against named or unnamed competitors or alternatives

Rules: every question must name "{BRAND}" (or a natural short form of it). Vary length and tone like real
users (some short and casual, some detailed situations). No numbering. Do not repeat these:
{json.dumps([t.format(brand=BRAND) for _, _, t in CORE_TEMPLATES])}"""
    out = _structured(WRITER_MODEL, "You write realistic consumer search questions. Return JSON only.",
                      user, "prompt_set", schema)
    seen, rows = set(), []
    for q in out["questions"]:
        key = q["prompt"].strip().lower()
        if key and key not in seen:
            seen.add(key)
            rows.append(q)
    return rows[:n]

# --- spend tracking (shared by all cells) ---
_spend_lock = threading.Lock()
SPEND = {"usd": 0.0}
def track_spend(usd):
    with _spend_lock:
        SPEND["usd"] += usd
def spent():
    with _spend_lock:
        return SPEND["usd"]

if REUSE_PROMPT_SET:
    from google.colab import files
    print("Upload a *_promptset.csv from a previous run…")
    up = files.upload()
    prompt_df = pd.read_csv(next(iter(up)))
    prompt_df["prompt"] = prompt_df.apply(
        lambda r: r["template"].format(brand=BRAND) if r["source"] == "core" else r["prompt"], axis=1)
    prompt_df = prompt_df[["prompt_id", "source", "intent", "template", "prompt"]]
else:
    n_core = min(len(CORE_TEMPLATES), ANALYSIS_DEPTH)
    rows = [{"prompt_id": pid, "source": "core", "intent": intent, "template": t,
             "prompt": t.format(brand=BRAND)} for pid, intent, t in CORE_TEMPLATES[:n_core]]
    n_extra = ANALYSIS_DEPTH - n_core
    if n_extra:
        print(f"Generating {n_extra} brand-specific questions with {WRITER_MODEL}…")
        for i, q in enumerate(generate_extra_prompts(n_extra), 1):
            rows.append({"prompt_id": f"gen-{i:03d}", "source": "generated", "intent": q["intent"],
                         "template": "", "prompt": q["prompt"].strip()})
    prompt_df = pd.DataFrame(rows)

print(f"✓ {len(prompt_df)} prompts  ·  " +
      "  ".join(f"{k}: {v}" for k, v in prompt_df["intent"].value_counts().items()))
display(prompt_df.sample(min(8, len(prompt_df)), random_state=1)[["prompt_id", "intent", "prompt"]])

In [ ]:
# @title 6 · Ask ChatGPT (collect answers)
def _citations(resp):
    cites = []
    for item in resp.output or []:
        if getattr(item, "type", "") != "message":
            continue
        for part in getattr(item, "content", None) or []:
            for a in getattr(part, "annotations", None) or []:
                if getattr(a, "type", "") == "url_citation":
                    cites.append({"url": a.url, "title": getattr(a, "title", "") or ""})
    uniq = {c["url"]: c for c in cites}
    return list(uniq.values())

def ask(prompt):
    kwargs = {"model": ANSWER_MODEL, "input": prompt}
    if USE_WEB_SEARCH:
        tool = {"type": "web_search"}
        loc = {k: v for k, v in {"country": MARKET_COUNTRY, "city": MARKET_CITY,
                                 "region": MARKET_REGION}.items() if v}
        if loc:
            tool["user_location"] = {"type": "approximate", **loc}
        kwargs["tools"] = [tool]
    resp = client.responses.create(**kwargs)
    searches = sum(1 for it in (resp.output or []) if getattr(it, "type", "") == "web_search_call")
    cost = usage_cost(resp.model or ANSWER_MODEL, resp.usage, searches)
    track_spend(cost)
    return {"response": (resp.output_text or "").strip(), "model_used": resp.model,
            "citations": _citations(resp), "search_calls": searches, "answer_cost": cost}

jobs = [(r, s) for r in prompt_df.to_dict("records") for s in range(1, SAMPLES_PER_PROMPT + 1)]
random.shuffle(jobs)  # spreads intents evenly if the budget cap stops the run early
answers, t0 = [], time.time()
avg_answer = est_call_cost(ANSWER_MODEL, "answer_search" if USE_WEB_SEARCH else "answer_nosearch")
judge_reserve = len(jobs) * est_call_cost(JUDGE_MODEL, "judge") + \
                (est_call_cost(WRITER_MODEL, "summary") if INCLUDE_THEME_SUMMARY else 0)
_inflight = {"n": 0}

def run_job(row, sample):
    with _spend_lock:
        if SPEND["usd"] + (_inflight["n"] + 1) * avg_answer + judge_reserve > MAX_BUDGET_USD:
            return {**row, "sample": sample, "response": None, "error": "skipped: budget cap"}
        _inflight["n"] += 1
    try:
        return {**row, "sample": sample, **ask(row["prompt"]), "error": None}
    except Exception as e:
        return {**row, "sample": sample, "response": None, "error": f"{type(e).__name__}: {e}"[:300]}
    finally:
        with _spend_lock:
            _inflight["n"] -= 1

print(f"Asking {len(jobs)} questions with {ANSWER_MODEL} ({MAX_WORKERS} at a time)…")
if len(jobs) * avg_answer + judge_reserve > MAX_BUDGET_USD:
    print(f"⚠️ Estimate exceeds ${MAX_BUDGET_USD:.2f} — the run will stop asking once the budget is reached "
          "(questions are shuffled, so the partial sample stays balanced).")
with ThreadPoolExecutor(MAX_WORKERS) as pool:
    futures = [pool.submit(run_job, r, s) for r, s in jobs]
    for i, f in enumerate(as_completed(futures), 1):
        answers.append(f.result())
        if i % 10 == 0 or i == len(jobs):
            print(f"  {i}/{len(jobs)} · ${spent():.2f} spent · {time.time() - t0:.0f}s")

ok = [a for a in answers if a["response"]]
errs = Counter(a["error"].split(":")[0] for a in answers if a["error"])
print(f"✓ {len(ok)}/{len(jobs)} answers" + (f"  ·  issues: {dict(errs)}" if errs else ""))
if ok:
    print(f"   Model actually used: {Counter(a['model_used'] for a in ok).most_common(1)[0][0]}")

In [ ]:
# @title 7 · Judge each answer (score, aspects, evidence)
ASPECTS = ["price_value", "quality", "customer_service", "trust_reputation", "results_effectiveness",
           "staff_expertise", "convenience_access", "innovation", "ethics_values", "other"]

JUDGE_SCHEMA = {
  "type": "object", "additionalProperties": False,
  "required": ["brand_knowledge", "score", "reason", "aspects", "positive_points", "negative_points"],
  "properties": {
    "brand_knowledge": {"type": "string", "enum": ["knows", "limited", "unknown", "wrong_entity"]},
    "score": {"type": "integer", "enum": [-2, -1, 0, 1, 2]},
    "reason": {"type": "string"},
    "aspects": {"type": "array", "items": {"type": "object", "additionalProperties": False,
        "required": ["aspect", "score"],
        "properties": {"aspect": {"type": "string", "enum": ASPECTS},
                       "score": {"type": "integer", "enum": [-2, -1, 0, 1, 2]}}}},
    "positive_points": {"type": "array", "items": {"type": "object", "additionalProperties": False,
        "required": ["point", "quote"], "properties": {"point": {"type": "string"}, "quote": {"type": "string"}}}},
    "negative_points": {"type": "array", "items": {"type": "object", "additionalProperties": False,
        "required": ["point", "quote"], "properties": {"point": {"type": "string"}, "quote": {"type": "string"}}}},
  }}

JUDGE_SYSTEM = """You rate how an AI assistant's answer portrays one specific brand to a consumer.
Score ONLY what the answer says about the target brand — ignore praise or criticism of competitors.

score:
 +2 clearly favorable: recommends it or praises it with little or no reservation
 +1 mostly favorable: positives outweigh minor caveats
  0 neutral/factual, or genuinely balanced
 -1 mostly unfavorable: concerns are emphasized or outweigh positives
 -2 clearly unfavorable: warns against it or highlights serious problems
Generic advice ("check reviews", "consult a professional") is neutral, not negative.

brand_knowledge:
 knows = gives specific information about this brand
 limited = only vague/general information
 unknown = says it has no information, or only talks generically about the category
 wrong_entity = describes a different company or person with a similar name
If unknown or wrong_entity, set score 0 and leave aspects/points empty.

positive_points / negative_points: each is a short plain-English point plus a VERBATIM quote
(copied exactly, 5–30 words) from the answer that supports it. Max 4 each. Use [] if none.
aspects: only aspects the answer actually evaluates. reason: one sentence."""

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[“”\"'‘’*_`]", "", s or "")).strip().lower()

def judge(row):
    ctx = f" (category: {CATEGORY})" if CATEGORY else ""
    user = (f"Target brand: {BRAND}{ctx}\n\nUser's question: {row['prompt']}\n\n"
            f"AI answer:\n<<<\n{row['response'][:12000]}\n>>>")
    j = _structured(JUDGE_MODEL, JUDGE_SYSTEM, user, "brand_judgment", JUDGE_SCHEMA, 1500)
    resp_n = _norm(row["response"])
    for side in ("positive_points", "negative_points"):
        for p in j[side]:
            p["verified"] = bool(p["quote"]) and _norm(p["quote"]) in resp_n
    return j

to_judge = [a for a in answers if a["response"]]
results = []
print(f"Judging {len(to_judge)} answers with {JUDGE_MODEL}…")
def _judge_job(a):
    try:
        return {**a, "judgment": judge(a), "judge_error": None}
    except Exception as e:
        return {**a, "judgment": None, "judge_error": f"{type(e).__name__}: {e}"[:300]}
with ThreadPoolExecutor(MAX_WORKERS) as pool:
    for i, r in enumerate(pool.map(_judge_job, to_judge), 1):
        results.append(r)
        if i % 20 == 0 or i == len(to_judge):
            print(f"  {i}/{len(to_judge)} · ${spent():.2f} spent")

def label_for(j):
    if j is None:
        return None
    if j["brand_knowledge"] in ("unknown", "wrong_entity"):
        return "unknown"
    has_pos, has_neg = bool(j["positive_points"]), bool(j["negative_points"])
    if j["score"] >= 1:
        return "positive"
    if j["score"] <= -1:
        return "negative"
    return "mixed" if (has_pos and has_neg) else "neutral"

for r in results:
    r["label"] = label_for(r["judgment"])
print(f"✓ Judged {sum(1 for r in results if r['judgment'])} · labels: "
      f"{dict(Counter(r['label'] for r in results if r['label']))}")

In [ ]:
# @title 8 · Scores & readout
def to_frame(results):
    rows = []
    for r in results:
        j = r.get("judgment") or {}
        rows.append({
            "run_id": RUN_ID, "run_date": RUN_DATE, "brand": BRAND, "category": CATEGORY,
            "answer_model": r.get("model_used") or ANSWER_MODEL, "web_search": USE_WEB_SEARCH,
            "prompt_id": r["prompt_id"], "source": r["source"], "intent": r["intent"],
            "template": r.get("template", ""), "prompt": r["prompt"], "sample": r["sample"],
            "response": r["response"], "brand_knowledge": j.get("brand_knowledge"),
            "score": j.get("score"), "label": r.get("label"), "reason": j.get("reason"),
            "aspects_json": json.dumps(j.get("aspects", [])),
            "positive_points_json": json.dumps(j.get("positive_points", [])),
            "negative_points_json": json.dumps(j.get("negative_points", [])),
            "citations_json": json.dumps(r.get("citations", [])),
            "cited_domains": "; ".join(sorted({urlparse(c["url"]).netloc.removeprefix("www.")
                                               for c in r.get("citations", [])})),
            "answer_cost_usd": round(r.get("answer_cost", 0) or 0, 5),
        })
    return pd.DataFrame(rows)

def score_runs(df, groupby=("brand", "run_id"), n_boot=1000, seed=7):
    """Sentiment Index 0–100 (50 = neutral). Samples are averaged per prompt first so every
    question counts equally; 90% CI by bootstrapping prompts. 'unknown' answers are excluded
    from the index and reported as recognition instead."""
    out = []
    rng = random.Random(seed)
    for keys, g in df.groupby(list(groupby)):
        judged = g[g["label"].notna()]
        known = judged[judged["label"] != "unknown"]
        per_prompt = known.groupby("prompt_id")["score"].mean()
        idx = lambda s: (s.mean() + 2) * 25 if len(s) else float("nan")
        boots = sorted(idx(per_prompt.sample(len(per_prompt), replace=True, random_state=rng.randrange(10**9)))
                       for _ in range(n_boot)) if len(per_prompt) > 1 else []
        rec = dict(zip(groupby, keys if isinstance(keys, tuple) else (keys,)))
        rec.update({
            "run_date": g["run_date"].iloc[0],
            "answers": len(judged),
            "sentiment_index": round(idx(per_prompt), 1),
            "ci90_low": round(boots[int(0.05 * n_boot)], 1) if boots else None,
            "ci90_high": round(boots[int(0.95 * n_boot)], 1) if boots else None,
            "brand_safe_rate": round(100 * (known["label"] != "negative").mean(), 1) if len(known) else None,
            "positive_pct": round(100 * (known["label"] == "positive").mean(), 1) if len(known) else None,
            "neutral_mixed_pct": round(100 * known["label"].isin(["neutral", "mixed"]).mean(), 1) if len(known) else None,
            "negative_pct": round(100 * (known["label"] == "negative").mean(), 1) if len(known) else None,
            "recognition_rate": round(100 * len(known) / len(judged), 1) if len(judged) else None,
            "core_index": round(idx(known[known["source"] == "core"].groupby("prompt_id")["score"].mean()), 1),
        })
        out.append(rec)
    return pd.DataFrame(out)

def aspect_table(df):
    rows = []
    for _, r in df[df["label"].notna() & (df["label"] != "unknown")].iterrows():
        for a in json.loads(r["aspects_json"]):
            rows.append(a)
    if not rows:
        return pd.DataFrame(columns=["aspect", "mentions", "aspect_index"])
    a = pd.DataFrame(rows).groupby("aspect")["score"].agg(["count", "mean"]).reset_index()
    a.columns = ["aspect", "mentions", "aspect_index"]
    a["aspect_index"] = ((a["aspect_index"] + 2) * 25).round(1)
    return a.sort_values("mentions", ascending=False)

def readout(df):
    s = score_runs(df).iloc[0]
    known = df[df["label"].notna() & (df["label"] != "unknown")]
    emoji = "😊" if s.sentiment_index >= 60 else "😞" if s.sentiment_index < 45 else "😐"
    print("=" * 64)
    print(f"AI BRAND SENTIMENT: {BRAND.upper()}")
    print(f"{RUN_DATE[:10]} · {s.answers} answers · {df['answer_model'].iloc[0]}"
          f"{' + web search' if USE_WEB_SEARCH else ''}")
    print("=" * 64)
    ci = f"  (90% CI {s.ci90_low}–{s.ci90_high})" if pd.notna(s.ci90_low) else ""
    print(f"\n{emoji} SENTIMENT INDEX: {s.sentiment_index}/100{ci}   ·  50 = neutral")
    print(f"🛡️  BRAND-SAFE RATE: {s.brand_safe_rate}% of answers are not negative (v1 metric)")
    print(f"🔎 RECOGNITION: AI gave brand-specific info in {s.recognition_rate}% of answers")
    print(f"📌 CORE-QUESTION INDEX: {s.core_index}  (use this one to compare brands)")
    print("\n📊 BREAKDOWN")
    for lab in ["positive", "neutral", "mixed", "negative", "unknown"]:
        n = (df["label"] == lab).sum()
        pct = 100 * n / max(1, df["label"].notna().sum())
        print(f"   {lab.capitalize():9}| {'█' * int(pct / 5):20} {pct:5.1f}% ({n})")
    print("\n🎯 BY QUESTION INTENT")
    for intent in INTENTS:
        g = known[known["intent"] == intent].groupby("prompt_id")["score"].mean()
        if len(g):
            print(f"   {intent:12} index {(g.mean() + 2) * 25:5.1f}   ({len(g)} questions)")
    at = aspect_table(df)
    if len(at):
        print("\n🧩 ASPECTS AI EVALUATES (index, mentions)")
        for _, a in at.head(8).iterrows():
            print(f"   {a.aspect:22} {a.aspect_index:5.1f}  ({a.mentions})")
    for side, title in (("negative_points_json", "❌ WHAT AI SAYS AGAINST"),
                        ("positive_points_json", "✅ WHAT AI SAYS IN FAVOR OF")):
        pts = list({_norm(p["quote"]): p for v in df[side].dropna()
                    for p in json.loads(v) if p.get("verified")}.values())
        if pts:
            print(f"\n{title} {BRAND.upper()} (verified quotes)")
            for p in pts[:4]:
                print(f"   • {p['point']}\n     “{p['quote'][:160]}”")
    unk = df[df["label"] == "unknown"]
    if len(unk) and (unk["brand_knowledge"] == "wrong_entity").any():
        print(f"\n⚠️  {(unk['brand_knowledge'] == 'wrong_entity').sum()} answer(s) described a DIFFERENT entity — check brand name/category")
    print(f"\n💵 Spend so far: ${spent():.2f}")
    print("=" * 64)
    return s, at

df = to_frame(results)
summary_row, aspects_df = readout(df)

# Charts
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
known = df[df["label"].notna() & (df["label"] != "unknown")]
by_intent = [(i, (known[known["intent"] == i].groupby("prompt_id")["score"].mean().mean() + 2) * 25)
             for i in INTENTS if (known["intent"] == i).any()]
axes[0].barh([i for i, _ in by_intent], [v for _, v in by_intent], color="#4C78A8")
axes[0].axvline(50, color="#888", ls="--", lw=1); axes[0].set_xlim(0, 100)
axes[0].set_title("Sentiment index by question intent"); axes[0].invert_yaxis()
if len(aspects_df):
    top = aspects_df.head(8)
    axes[1].barh(top["aspect"], top["aspect_index"],
                 color=["#54A24B" if v >= 55 else "#E45756" if v < 45 else "#BAB0AC" for v in top["aspect_index"]])
    axes[1].axvline(50, color="#888", ls="--", lw=1); axes[1].set_xlim(0, 100); axes[1].invert_yaxis()
axes[1].set_title("Sentiment index by aspect")
plt.tight_layout(); plt.show()

In [ ]:
# @title 9 · Theme summary: what's driving positive & negative sentiment (optional)
# One extra call to WRITER_MODEL over the judged points (not full answers): ~$0.05–0.10, ~20–40 s.
THEME_SCHEMA = {
  "type": "object", "additionalProperties": False,
  "required": ["headline", "positive_themes", "negative_themes", "watch_items"],
  "properties": {
    "headline": {"type": "string"},
    "positive_themes": {"type": "array", "items": {"$ref": "#/$defs/theme"}},
    "negative_themes": {"type": "array", "items": {"$ref": "#/$defs/theme"}},
    "watch_items": {"type": "array", "items": {"type": "string"}},
  },
  "$defs": {"theme": {"type": "object", "additionalProperties": False,
    "required": ["theme", "answers_count", "explanation", "evidence_ids", "likely_sources", "suggested_action"],
    "properties": {"theme": {"type": "string"}, "answers_count": {"type": "integer"},
                   "explanation": {"type": "string"},
                   "evidence_ids": {"type": "array", "items": {"type": "string"}},
                   "likely_sources": {"type": "array", "items": {"type": "string"}},
                   "suggested_action": {"type": "string"}}}},
}

def build_evidence(df):
    ev = []
    for i, r in df[df["label"].notna() & (df["label"] != "unknown")].iterrows():
        for side, key in (("pos", "positive_points_json"), ("neg", "negative_points_json")):
            for k, p in enumerate(json.loads(r[key])):
                ev.append({"id": f"{side}-{i}-{k}", "side": side, "intent": r["intent"],
                           "point": p["point"], "quote": p["quote"] if p.get("verified") else "",
                           "domains": r["cited_domains"]})
    return ev

def summarize_themes(df):
    ev = build_evidence(df)
    if not ev:
        return None, ev
    user = (f"Brand: {BRAND}. Below are points an AI assistant made about the brand across "
            f"{df['label'].notna().sum()} answers, each with the web domains the answer cited.\n"
            "Group them into recurring themes (max 6 per side), count how many distinct answers "
            "raise each theme, cite evidence ids, name the domains most associated with each theme, "
            "and give one practical action for each (for negatives: what content, reviews or "
            "corrections could shift how AI describes this; for positives: how to reinforce it). "
            "headline: one sentence a marketer could put on a slide. watch_items: emerging or "
            "inaccurate-looking claims worth verifying.\n\n" + json.dumps(ev)[:120000])
    out = _structured(WRITER_MODEL, "You are a brand reputation analyst. Be specific and concise. JSON only.",
                      user, "theme_summary", THEME_SCHEMA, 5000)
    return out, ev

THEMES = None
if INCLUDE_THEME_SUMMARY:
    t0 = time.time()
    THEMES, EVIDENCE = summarize_themes(df)
    if THEMES:
        ev_by_id = {e["id"]: e for e in EVIDENCE}
        print(f"📝 {THEMES['headline']}\n")
        for side, title in (("negative_themes", "❌ NEGATIVE THEMES"), ("positive_themes", "✅ POSITIVE THEMES")):
            print(title)
            for t in THEMES[side]:
                print(f"\n  ▸ {t['theme']}  ({t['answers_count']} answers)")
                print(f"    {t['explanation']}")
                q = next((ev_by_id[e]["quote"] for e in t["evidence_ids"] if ev_by_id.get(e, {}).get("quote")), None)
                if q:
                    print(f"    e.g. “{q[:180]}”")
                if t["likely_sources"]:
                    print(f"    Sources: {', '.join(t['likely_sources'][:5])}")
                print(f"    → {t['suggested_action']}")
            print()
        if THEMES["watch_items"]:
            print("👀 WATCH / VERIFY")
            for w in THEMES["watch_items"]:
                print(f"   • {w}")
        print(f"\n(theme summary took {time.time() - t0:.0f}s · total spend ${spent():.2f})")
    else:
        print("No positive or negative points to summarize.")
else:
    print("Theme summary skipped (INCLUDE_THEME_SUMMARY = False)")

In [ ]:
# @title 10 · Export CSVs (responses, run summary, prompt set)
def export_run(df, download=True):
    base = f"{BRAND_SLUG}_{RUN_ID}"
    paths = {
        "responses": f"{base}_responses.csv",
        "summary":   f"{base}_summary.csv",
        "promptset": f"{base}_promptset.csv",
    }
    df.to_csv(paths["responses"], index=False)
    summ = score_runs(df)
    summ["total_cost_usd"] = round(spent(), 3)
    summ["answer_model"] = df["answer_model"].iloc[0]
    summ["web_search"] = USE_WEB_SEARCH
    summ["samples_per_prompt"] = SAMPLES_PER_PROMPT
    if THEMES:
        summ["headline"] = THEMES["headline"]
        summ["themes_json"] = json.dumps(THEMES)
    summ.to_csv(paths["summary"], index=False)
    prompt_df.to_csv(paths["promptset"], index=False)
    print("Saved:\n" + "\n".join(f"   {p}" for p in paths.values()))
    if download:
        try:
            from google.colab import files
            for p in paths.values():
                files.download(p)
        except Exception:
            pass
    return paths

EXPORTED = export_run(df)
print("\nKeep the *_responses.csv — upload it in cell 11 later to track this brand over time "
      "or compare it with other brands.")

In [ ]:
# @title 11 · Track over time / compare brands (upload previous *_responses.csv files)
INCLUDE_CURRENT_RUN = True  # @param {type:"boolean"}
UPLOAD_FILES = True  # @param {type:"boolean"}

frames = [df] if INCLUDE_CURRENT_RUN and "df" in globals() else []
if UPLOAD_FILES:
    from google.colab import files
    print("Select one or more *_responses.csv files from earlier runs (any brands)…")
    for name, data in files.upload().items():
        import io
        frames.append(pd.read_csv(io.BytesIO(data)))
        print(f"   loaded {name}")

if not frames:
    print("Nothing to compare yet.")
else:
    hist = pd.concat(frames, ignore_index=True).drop_duplicates(["run_id", "prompt_id", "sample"])
    runs = score_runs(hist).sort_values(["brand", "run_date"])
    cols = ["brand", "run_date", "answers", "sentiment_index", "ci90_low", "ci90_high",
            "core_index", "brand_safe_rate", "recognition_rate", "negative_pct"]
    display(runs[cols])

    # Core questions are identical across runs and brands → the fair comparison
    core = hist[(hist["source"] == "core") & hist["label"].notna() & (hist["label"] != "unknown")]
    if len(core):
        pivot = (core.groupby(["template", "brand", "run_id"])["score"].mean().add(2).mul(25).round(0)
                     .unstack(["brand", "run_id"]))
        print("\nCore questions, index by run (same wording for every brand):")
        display(pivot)

    fig, ax = plt.subplots(figsize=(10, 4))
    for brand, g in runs.groupby("brand"):
        x = pd.to_datetime(g["run_date"])
        ax.plot(x, g["core_index"], marker="o", label=brand)
        if g["ci90_low"].notna().all():
            ax.fill_between(x, g["ci90_low"], g["ci90_high"], alpha=0.12)
    ax.axhline(50, color="#888", ls="--", lw=1)
    ax.set_ylim(0, 100); ax.set_ylabel("Core sentiment index"); ax.legend()
    ax.set_title("AI sentiment over time (core questions)")
    plt.tight_layout(); plt.show()

    if hist["answer_model"].nunique() > 1:
        print("ℹ️ Runs used different ChatGPT models — some movement may come from model updates, not the brand.")